## Pull ACS age distributions for select cities
Goal: To get jurisdiction level age distributions from select years
Method for ETL: 
- Pull data from acs 1-year data.
- Recode age bins to five year groups


In [ ]:
import os
census_api_key = os.getenv('CENSUS_API_KEY')
import pandas as pd
import geopandas as gpd
import numpy as np
import os
import pathlib
import glob
HOME_DIR = pathlib.Path.home()
# in my (AO) case, M:\urban_modeling is mounted to /Volumes/Data/Models/urban_modeling
m_path = M_DRIVE = pathlib.Path(
    "/Volumes/Data/Models") if os.name != "nt" else pathlib.Path("M:/")

box_path = HOME_DIR / 'Library/CloudStorage/Box-Box'
m_path

In [8]:
import requests
import pandas as pd
from functools import lru_cache

# (state FIPS, place FIPS) for each jurisdiction's incorporated place geography
CITY_GEO = {
    "San Francisco": ("06", "67000"),
    "Los Angeles":   ("06", "44000"),
    "New York":      ("36", "51000"),
    "Austin":        ("48", "05000"),
    "Chicago":       ("17", "14000"),
    "Seattle":       ("53", "63000"),
    "Portland":      ("41", "59000"),
}


@lru_cache(maxsize=None)
def _b01001_labels(year, dataset="acs/acs5"):
    """Fetch & cache variable labels for B01001 (code -> label) for a given vintage."""
    url = f"https://api.census.gov/data/{year}/{dataset}/groups/B01001.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    variables = resp.json()["variables"]
    return {code: v["label"] for code, v in variables.items() if code.startswith("B01001_") and code.endswith("E")}


def _parse_b01001_label(label):
    # label looks like "Estimate!!Total:" / "...!!Male:" / "...!!Male:!!Under 5 years"
    tokens = [t.rstrip(":") for t in label.split("!!")[1:]]
    if len(tokens) == 1:
        return "Total", "All ages"
    if len(tokens) == 2:
        return tokens[1], "All ages"
    return tokens[1], tokens[2]


In [44]:
def fetch_acs_b01001(cities, year, api_key=None, dataset="acs/acs1"):
    """
    Pull ACS table B01001 (sex by age) for one or more cities (jurisdiction/place level).

    Parameters
    ----------
    cities : str or list[str]
        City name(s), must be keys in CITY_GEO.
    year : int
        ACS vintage year, e.g. 2022.
    api_key : str, optional
        Census API key (recommended to avoid rate limiting).
    dataset : str
        Census dataset, defaults to 5-year ACS detailed tables.

    Returns
    -------
    pandas.DataFrame
        Tidy dataframe: city, year, variable, sex, age_group, estimate
    """
    if isinstance(cities, str):
        cities = [cities]
    unknown = sorted(set(cities) - set(CITY_GEO))
    if unknown:
        raise ValueError(f"Unknown city(ies): {unknown}. Add them to CITY_GEO.")

    # group requested cities by state so each state only needs one API call
    by_state = {}
    for city in cities:
        state, place = CITY_GEO[city]
        by_state.setdefault(state, []).append((city, place))

    labels = _b01001_labels(year, dataset)
    base_url = f"https://api.census.gov/data/{year}/{dataset}"
    frames = []
    with requests.Session() as session:
        for state, items in by_state.items():
            params = {
                "get": "NAME,group(B01001)",
                "for": f"place:{','.join(place for _, place in items)}",
                "in": f"state:{state}",
            }
            if api_key:
                params["key"] = api_key
            resp = session.get(base_url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            frames.append(pd.DataFrame(data[1:], columns=data[0]))

    wide = pd.concat(frames, ignore_index=True)
    place_to_city = {place: city for city, (_, place) in CITY_GEO.items() if city in cities}
    wide["city"] = wide["place"].map(place_to_city)

    est_cols = [c for c in wide.columns if c.startswith("B01001_") and c.endswith("E")]
    long = wide.melt(id_vars=["city"], value_vars=est_cols, var_name="variable", value_name="estimate")
    long["estimate"] = pd.to_numeric(long["estimate"], errors="coerce")
    long["year"] = year
    long[["sex", "age_group"]] = long["variable"].map(labels).apply(_parse_b01001_label).apply(pd.Series)

    return long[["city", "year", "variable", "sex", "age_group", "estimate"]].sort_values(
        ["city", "variable"]).reset_index(drop=True)


In [45]:
cities = ["San Francisco", "Los Angeles", "New York", "Austin", "Chicago", "Seattle", "Portland"]
df_2021 = fetch_acs_b01001(cities, year=2021, api_key=census_api_key)
df_2022 = fetch_acs_b01001(cities, year=2022, api_key=census_api_key)
df_2023 = fetch_acs_b01001(cities, year=2023, api_key=census_api_key)
df_2024 = fetch_acs_b01001(cities, year=2024, api_key=census_api_key)
df_2022.head(10)


,city,year,variable,sex,age_group,estimate
0,Austin,2022,B01001_001E,Total,All ages,975335
1,Austin,2022,B01001_002E,Male,All ages,497378
2,Austin,2022,B01001_003E,Male,Under 5 years,24206
3,Austin,2022,B01001_004E,Male,5 to 9 years,23201
4,Austin,2022,B01001_005E,Male,10 to 14 years,21508
5,Austin,2022,B01001_006E,Male,15 to 17 years,15112
6,Austin,2022,B01001_007E,Male,18 and 19 years,14808
7,Austin,2022,B01001_008E,Male,20 years,5822
8,Austin,2022,B01001_009E,Male,21 years,5706
9,Austin,2022,B01001_010E,Male,22 to 24 years,26342


In [46]:
df_2024.query('city=="San Francisco" & year==2024')

,city,year,variable,sex,age_group,estimate
245,San Francisco,2024,B01001_001E,Total,All ages,827526
246,San Francisco,2024,B01001_002E,Male,All ages,425910
247,San Francisco,2024,B01001_003E,Male,Under 5 years,16865
248,San Francisco,2024,B01001_004E,Male,5 to 9 years,12268
249,San Francisco,2024,B01001_005E,Male,10 to 14 years,18378
250,San Francisco,2024,B01001_006E,Male,15 to 17 years,9996
251,San Francisco,2024,B01001_007E,Male,18 and 19 years,6360
252,San Francisco,2024,B01001_008E,Male,20 years,3493
253,San Francisco,2024,B01001_009E,Male,21 years,2783
254,San Francisco,2024,B01001_010E,Male,22 to 24 years,15145


In [47]:
df[df["variable"] == "B01001_001E"][["city", "estimate"]]


,city,estimate
0,Austin,958202
49,Chicago,2721914
98,Los Angeles,3881041
147,New York,8622467
196,Portland,646101
245,San Francisco,851036
294,Seattle,734603


In [76]:
combo = pd.concat([df_2021, df_2022, df_2023, df_2024], ignore_index=True)

# add coding of age_group into age group 5
# first we need a mapping as the groups are currently uneven (B01001's actual bins, not even 5-year spans)
cutoff_age_map = {'Under 5 years':4,
                  '5 to 9 years':9,
                  '10 to 14 years':14,
                  '15 to 17 years':17,
                  '18 and 19 years':19,
                  '20 years':20,
                  '21 years':21,
                  '22 to 24 years':24,
                  '25 to 29 years':29,
                  '30 to 34 years':34,
                  '35 to 39 years':39,
                  '40 to 44 years':44,
                  '45 to 49 years':49,
                  '50 to 54 years':54,
                  '55 to 59 years':59,
                  '60 and 61 years':61,
                  '62 to 64 years':64,
                  '65 and 66 years':66,
                  '67 to 69 years':69,
                  '70 to 74 years':74,
                  '75 to 79 years':79,
                  '80 to 84 years':84,
                  '85 years and over':85
                  }

# 'All ages' rows are Total/Male/Female subtotals, not an age bin, so they stay unmapped (NaN)
# just remove the totals
combo = combo[combo['age_group'] != 'All ages']
unmapped = set(combo['age_group'].unique()) - set(cutoff_age_map) - {'All ages'}
assert not unmapped, f"cutoff_age_map is missing age_group value(s): {unmapped}"

combo['cutoff_age'] = combo['age_group'].map(cutoff_age_map)

# use the cutoffs to re-bin into five year groups
# this will create age groups like 0-4, 5-9, 10-14, etc.
# be sure to get the range in the string name
combo['age_group_5'] = (combo['cutoff_age'] // 5) * 5
combo['age_group_5'] = combo['age_group_5'].astype(str) + '-' + (combo['age_group_5'] + 4).astype(str)
# '85 years and over' is top-coded, not a true 5-year span, so relabel it '85+'
top_bin = max((v // 5) * 5 for v in cutoff_age_map.values())
combo['age_group_5'] = combo['age_group_5'].replace(f'{top_bin}-{top_bin + 4}', f'{top_bin}+')

# turn age_group_5 to a categorical to maintain proper sort order
age_order = [f'{b}-{b + 4}' for b in sorted({(v // 5) * 5 for v in cutoff_age_map.values()})]
age_order[-1] = f'{top_bin}+'
combo['age_group_5'] = pd.Categorical(combo['age_group_5'], categories=age_order, ordered=True)

In [73]:
# Sanity check / inspect for SF 
combo.query('city=="San Francisco"').head(10)

,city,year,variable,sex,age_group,estimate,cutoff_age,age_group_5
247,San Francisco,2021,B01001_003E,Male,Under 5 years,18170,4,0-4
248,San Francisco,2021,B01001_004E,Male,5 to 9 years,15870,9,5-9
249,San Francisco,2021,B01001_005E,Male,10 to 14 years,15781,14,10-14
250,San Francisco,2021,B01001_006E,Male,15 to 17 years,8864,17,15-19
251,San Francisco,2021,B01001_007E,Male,18 and 19 years,5522,19,15-19
252,San Francisco,2021,B01001_008E,Male,20 years,1999,20,20-24
253,San Francisco,2021,B01001_009E,Male,21 years,2341,21,20-24
254,San Francisco,2021,B01001_010E,Male,22 to 24 years,15965,24,20-24
255,San Francisco,2021,B01001_011E,Male,25 to 29 years,37551,29,25-29
256,San Francisco,2021,B01001_012E,Male,30 to 34 years,47801,34,30-34


In [77]:
# create new frame aggregating counts by five year age groups, year, variable, and city
agg_combo = combo.groupby(['age_group_5', 'year', 'sex','city'], as_index=False)['estimate'].sum()
agg_combo.query('city=="San Francisco" & year==2024').sample(10)

/var/folders/03/fgvd1kc91871lp3_mr1g8hlw0000gq/T/ipykernel_65799/568809920.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_combo = combo.groupby(['age_group_5', 'year', 'sex','city'], as_index=False)['estimate'].sum()


,age_group_5,year,sex,city,estimate
327,25-29,2024,Female,San Francisco,37616
54,0-4,2024,Male,San Francisco,16865
1006,85+,2024,Male,San Francisco,8276
334,25-29,2024,Male,San Francisco,39273
670,55-59,2024,Male,San Francisco,26401
278,20-24,2024,Male,San Francisco,21421
775,65-69,2024,Female,San Francisco,20498
159,10-14,2024,Female,San Francisco,16044
446,35-39,2024,Male,San Francisco,41651
47,0-4,2024,Female,San Francisco,15517


In [81]:
# example table for San Francisco by year, age group, and sex
agg_combo.query('city=="San Francisco"').set_index(['year','agoh e_group_5','sex']).estimate.unstack(['year','sex'])

year          2021          2022          2023          2024       
sex         Female   Male Female   Male Female   Male Female   Male
age_group_5                                                        
0-4          16662  18170  16973  16356  15288  16908  15517  16865
5-9          14181  15870  14670  13977  14880  15481  13387  12268
10-14        15615  15781  14099  16683  14180  15231  16044  18378
15-19        16178  14386  15178  14562  15673  15921  16513  16356
20-24        20173  20305  21387  20203  19602  18890  21381  21421
25-29        36609  37551  34912  38071  36020  37400  37616  39273
30-34        42495  47801  41090  46933  40717  45556  40675  45982
35-39        31599  37856  35764  38853  34180  37762  35091  41651
40-44        30399  32753  25080  32155  27041  34015  28029  33470
45-49        24948  28579  23912  27112  24497  28083  24702  27528
50-54        25269  29799  24863  29945  24721  29126  24902  28616
55-59        24413  28941  23600  28178  21850  26014  23093  26401
60-64        22463  23923  21847  24293  23625  26661  22588  26950
65-69        21227  19802  20846  21210  21796  22521  20498  22057
70-74        20677  20102  21400  18648  20731  17839  21698  19327
75-79        10784  11312  14402  12698  15356  12875  18627  14268
80-84         9987   7266  10425   7166   8322   6845   8937   6823
85+          14209   7116  13298   7648  14449   8932  12318   8276

## Export workflow 
Create workbook, loooping over cities in dataframe.

In [97]:
def build_city_table(city, agg_combo):
    """Pivot age_group_5 x (year, sex) estimates for one city; adds Male+Female Total columns/row."""
    sub = agg_combo[agg_combo['city'] == city]
    years = sorted(sub['year'].unique())
    table = sub.pivot_table(index='age_group_5', columns=['year', 'sex'], values='estimate', observed=True)
    for year in years:
        table[(year, 'Total')] = table[(year, 'Male')] + table[(year, 'Female')]
    table = table.reindex(columns=pd.MultiIndex.from_product([years, ['Male', 'Female', 'Total']]))
    table = table.sort_index()
    table.loc['All ages'] = table.sum()
    return table



In [98]:
from datetime import date
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
HEADER_FONT = Font(color='FFFFFF', bold=True)
TITLE_FONT = Font(bold=True, size=14)
SECTION_FONT = Font(bold=True)
TOTAL_FONT = Font(bold=True)
TOTAL_FILL = PatternFill(start_color='D9E1F2', end_color='D9E1F2', fill_type='solid')
HEADER_BOTTOM_BORDER = Border(bottom=Side(style='thin', color='B7B7B7'))
CENTER = Alignment(horizontal='center', vertical='center')

# table written via to_excel(startrow=TABLE_START_ROW) always has 3 header rows: year, sex, index name
TABLE_START_ROW = 3
N_HEADER_ROWS = 3


def _autofit_columns(ws, last_col):
    for col_idx in range(1, last_col + 1):
        letter = get_column_letter(col_idx)
        # skip row 1: the merged title cell would otherwise blow out column A's width
        lengths = [len(str(ws.cell(row=r, column=col_idx).value))
                   for r in range(2, ws.max_row + 1) if ws.cell(row=r, column=col_idx).value is not None]
        ws.column_dimensions[letter].width = (max(lengths) if lengths else 8) + 2


def _style_city_sheet(ws, city, n_years):
    n_cols = 1 + n_years * 3  # index column + (Male, Female, Total) per year
    year_row, sex_row, idx_name_row = TABLE_START_ROW, TABLE_START_ROW + 1, TABLE_START_ROW + 2
    data_start_row = idx_name_row + 1

    ws.cell(row=1, column=1, value=f'{city} \u2014 Population by Age and Sex (ACS Table B01001)').font = TITLE_FONT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)

    for r in (year_row, sex_row):
        for c in range(1, n_cols + 1):
            cell = ws.cell(row=r, column=c)
            cell.font = HEADER_FONT
            cell.fill = HEADER_FILL
            cell.alignment = CENTER
    for c in range(1, n_cols + 1):
        ws.cell(row=idx_name_row, column=c).border = HEADER_BOTTOM_BORDER

    for r in range(data_start_row, ws.max_row + 1):
        for c in range(2, n_cols + 1):
            ws.cell(row=r, column=c).number_format = '#,##0'

    for c in range(1, n_cols + 1):
        cell = ws.cell(row=ws.max_row, column=c)
        cell.font = TOTAL_FONT
        cell.fill = TOTAL_FILL

    ws.freeze_panes = ws.cell(row=data_start_row, column=2).coordinate
    _autofit_columns(ws, n_cols)


def _write_readme(wb, cities, years):
    ws = wb.create_sheet('README', 0)
    rows = [
        ('Population by Age and Sex \u2014 ACS Table B01001', TITLE_FONT),
        ('', None),
        (f'Generated: {date.today().isoformat()}', None),
        ('Source: U.S. Census Bureau, American Community Survey, Table B01001 (Sex by Age)', None),
        ('Geography: incorporated place / jurisdiction level (not metro or county)', None),
        (f'ACS vintages: {", ".join(str(y) for y in years)}', None),
        (f'Cities: {", ".join(cities)}', None),
        ('', None),
        ('Sheet layout', SECTION_FONT),
        ('One sheet per city. Rows are 5-year age groups, youngest to oldest; "85+" is top-coded '
         '(ACS reports "85 years and over" as a single open-ended group, not a true 5-year span).', None),
        ('Columns are grouped by ACS vintage year, then Male / Female / Total. The bottom "All ages" '
         'row sums every age group for that city/year/sex.', None),
        ('', None),
        ('Notes', SECTION_FONT),
        ('"Total" columns and the "All ages" row are Male + Female, computed locally \u2014 not a '
         'separate Census variable.', None),
        ('ACS estimates carry margins of error not shown here; re-query the Census API for *_M fields if needed.', None),
    ]
    for i, (text, font) in enumerate(rows, start=1):
        cell = ws.cell(row=i, column=1, value=text)
        cell.alignment = Alignment(wrap_text=True, vertical='top')
        if font:
            cell.font = font
    ws.column_dimensions['A'].width = 110


def export_workbook(agg_combo, cities, output_path):
    """Write one formatted sheet per city plus a README sheet to output_path."""
    years = sorted(agg_combo['year'].unique())
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for city in cities:
            build_city_table(city, agg_combo).to_excel(writer, sheet_name=city[:31], startrow=TABLE_START_ROW - 1)
        wb = writer.book
        for city in cities:
            _style_city_sheet(wb[city[:31]], city, n_years=len(years))
        _write_readme(wb, cities, years)
    return output_path


In [90]:
output_folder = M_DRIVE / 'Data/Census/Summaries'
if not output_folder.is_dir():
    print(f"{output_folder} not reachable (M: drive not mounted?) \u2014 writing locally instead.")
    output_folder = pathlib.Path('.')
export_workbook(agg_combo, cities, output_folder / 'age_distribution_combo.xlsx')


PosixPath('/Volumes/Data/Models/Data/Census/Summaries/age_distribution_combo.xlsx')

In [99]:
# # check total population for San Francisco in ACS 5-year and 1-year estimates
# acs5_sf = fetch_acs_b01001("San Francisco", year=2024, api_key=census_api_key, dataset="acs/acs5")
# acs1_sf = fetch_acs_b01001("San Francisco", year=2024, api_key=census_api_key, dataset="acs/acs1")
# print("acs5 total:", acs5_sf.loc[acs5_sf["variable"] == "B01001_001E", "estimate"].iloc[0])
# print("acs1 total:", acs1_sf.loc[acs1_sf["variable"] == "B01001_001E", "estimate"].iloc[0])


## Plot pyramids for select cities and / or years

In [94]:
import plotly.express as px


def _pretty(name):
    """'age_group_5' -> 'Age Group 5'"""
    return name.replace('_', ' ').title()


def summarize_population_share(df, group_col, category_col, facet_col=None, value_col='estimate'):
    """Share of `category_col` within each `group_col` (and `facet_col`, if given), from pre-aggregated estimates."""
    group_keys = [group_col, category_col] + ([facet_col] if facet_col else [])
    denom_keys = [group_col] + ([facet_col] if facet_col else [])

    numer = df.groupby(group_keys, observed=True)[value_col].sum().rename('n').reset_index()
    denom = df.groupby(denom_keys, observed=True)[value_col].sum().rename('total').reset_index()
    share_df = numer.merge(denom, on=denom_keys)
    share_df['share'] = share_df['n'] / share_df['total']
    return share_df


def plot_population_pyramid(share_df, group_col, category_col, facet_col=None, category_order=None, left_value='Male'):
    """Diverging horizontal bar (population pyramid) built from a summarize_population_share() result."""
    plot_df = share_df.copy()
    plot_df['signed_share'] = np.where(plot_df[group_col] == left_value, -plot_df['share'], plot_df['share'])

    title = f'Population Pyramid by {_pretty(category_col)} and {_pretty(group_col)}'
    if facet_col:
        title += f', by {_pretty(facet_col)}'

    fig = px.bar(
        plot_df,
        x='signed_share',
        y=category_col,
        color=group_col,
        orientation='h',
        facet_col=facet_col,
        # plotly reverses the y category order it's given for horizontal bars, so un-reverse here
        # to let callers pass category_order bottom-to-top (e.g. youngest to oldest)
        category_orders={category_col: list(reversed(category_order))} if category_order else None,
        text=plot_df['share'].map(lambda x: f'{x:.01%}'),
        labels={'signed_share': 'Share of Population', category_col: _pretty(category_col), group_col: _pretty(group_col)},
        title=title,
    )

    max_share = plot_df['share'].max()
    tick_vals = np.linspace(-max_share, max_share, 5)
    fig.update_xaxes(tickvals=tick_vals, ticktext=[f'{abs(v):.0%}' for v in tick_vals])
    fig.update_layout(barmode='relative', legend_title_text=_pretty(group_col), width=900, height=600)
    fig.update_traces(
        textposition='inside',
        hovertemplate=f'{_pretty(group_col)}: %{{fullData.name}}<br>{_pretty(category_col)}: %{{y}}<br>Share: %{{text}}<extra></extra>',
    )
    if facet_col:
        # strip the "col=value" prefix plotly adds to facet subplot titles
        fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))

    return fig


def plot_age_pyramid(df, city, years=None, group_col='sex', category_col='age_group_5',
                      value_col='estimate', category_order=age_order, left_value='Male'):
    """Population pyramid for one or more cities and/or years.

    Pass a list to `city` to facet by city (optionally still filtering to one or more `years`),
    or pass a list to `years` to facet by year (for a single city). Only one of the two should
    be a multi-item list at a time.
    """
    cities = [city] if isinstance(city, str) else list(city)
    subset = df[df['city'].isin(cities)]

    if years is not None:
        year_list = [years] if isinstance(years, int) else list(years)
        subset = subset[subset['year'].isin(year_list)]
    else:
        year_list = sorted(subset['year'].unique())

    facet_col = 'city' if len(cities) > 1 else ('year' if len(year_list) > 1 else None)

    share_df = summarize_population_share(subset, group_col, category_col, facet_col=facet_col, value_col=value_col)
    return plot_population_pyramid(share_df, group_col, category_col, facet_col=facet_col,
                                    category_order=category_order, left_value=left_value)

In [95]:
# facet by year, for a single city
fig = plot_age_pyramid(agg_combo, city="San Francisco", years=[2021, 2024])
fig.show()

In [96]:
# facet by city, for a single year
fig = plot_age_pyramid(agg_combo, city=["San Francisco", "Los Angeles", "Austin"], years=2024)
fig.show()